# **Time Series Forecasting with Facebook Prophet**

## Introduction
In this exercise, you will use **Facebook Prophet** to forecast Microsoft's closing stock price.

You will:
1. Load and preprocess the data into Prophet's required format
2. Initialize, configure, and fit a Prophet model
3. Generate future forecasts and visualise results
4. Evaluate forecast accuracy with Mean Absolute Error (MAE)

Fill in every `________` blank to complete the code.

**Dataset:** [Microsoft Stock — Time Series Analysis](https://www.kaggle.com/datasets/vijayvvenkitesh/microsoft-stock-time-series-analysis) — download `Microsoft_Stock.csv` and place it in the same directory as this notebook.

The dataset contains daily OHLCV data for Microsoft (MSFT) with columns: `Date`, `Open`, `High`, `Low`, `Close`, `Volume`.

## Step 1: Import Libraries

We need `pandas` for data handling, `matplotlib` for plotting, and `Prophet` for time series forecasting.

**Task:** Import `Prophet` from the correct module.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from prophet import Prophet


## Step 2: Load and Preprocess the Dataset

Prophet requires a DataFrame with exactly two columns:
- `ds` — the datestamp column
- `y` — the target value to forecast

We will forecast the **closing price** of Microsoft stock.

**Hint:** The relevant columns in this dataset are `Date` and `Close`.

In [ ]:
df = pd.read_csv("Microsoft_Stock.csv")

df.head()


In [ ]:
df = df[["Date", "Close"]]
df.columns = ["ds", "y"]


In [ ]:
# Ensure 'ds' is datetime type
df['ds'] = pd.to_datetime(df['ds'])

# Sort by date
df = df.sort_values('ds').reset_index(drop=True)

print(f"Date range: {df['ds'].min()} to {df['ds'].max()}")
print(f"Number of data points: {len(df)}")

In [ ]:
# Check for missing values
print(df.isnull().sum())

# Drop missing values if any
df = df.dropna()

In [ ]:
# Visualise the historical closing price
plt.figure(figsize=(12, 4))
plt.plot(df['ds'], df['y'])
plt.xlabel('Date')
plt.ylabel('Close Price (USD)')
plt.title('Microsoft (MSFT) — Daily Closing Price')
plt.tight_layout()
plt.show()

## Step 3: Initialize and Fit the Prophet Model

Prophet accepts several configuration parameters. A common one is `yearly_seasonality`, which tells the model whether to fit a yearly seasonal component.

**Task:** Create a Prophet model with `yearly_seasonality` set to `True`, then fit it to the data.

In [ ]:
model = Prophet(yearly_seasonality=True)

model.fit(df)


## Step 4: Generate the Forecast

We create a dataframe of future dates and use the trained model to predict.

**Task:** Generate a future dataframe covering 90 days beyond the training data.

**Note:** Stock markets are closed on weekends. By default, `make_future_dataframe` generates
calendar days (including weekends). We can pass `freq='B'` to generate only **business days**.

In [ ]:
future = model.make_future_dataframe(periods=90, freq='B')

print(f"Future dataframe shape: {future.shape}")
future.tail()


In [ ]:
# Generate the forecast
forecast = model.predict(future)

# Inspect key forecast columns
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

## Step 5: Visualise the Forecast and Components

In [ ]:
# Plot the forecast (black dots = actual, blue line = prediction, shaded = uncertainty)
fig = model.plot(forecast)
plt.title('Prophet Forecast — Microsoft (MSFT) Closing Price')
plt.ylabel('Close Price (USD)')
plt.show()

In [ ]:
# Plot trend + seasonal components
fig2 = model.plot_components(forecast)
plt.show()

## Step 6: Evaluate Forecast Accuracy

We compute the **Mean Absolute Error (MAE)** on the historical (in-sample) predictions.

In [ ]:
from sklearn.metrics import mean_absolute_error

# Merge actual and predicted values
df_forecast = forecast[['ds', 'yhat']].set_index('ds')
df_actual = df.set_index('ds')
df_combined = pd.merge(df_actual, df_forecast, left_index=True, right_index=True)

# Calculate Mean Absolute Error
mae = mean_absolute_error(df_combined['y'], df_combined['yhat'])
print(f'Mean Absolute Error: ${mae:,.2f}')